In [0]:
CREATE OR REPLACE TABLE jovi_lakehouse.gold.promoter_productivity AS

WITH parameters AS (
    SELECT
        CURRENT_DATE() AS today,
        CURRENT_DATE() - INTERVAL 14 DAY AS start_last_15_days,
        CURRENT_DATE() - INTERVAL 29 DAY AS start_last_30_days,
        CURRENT_DATE() - INTERVAL 59 DAY AS start_last_60_days,
        CURRENT_DATE() - INTERVAL 89 DAY AS start_last_90_days
),

base AS (
    SELECT
        p.id,
        p.nome_colaborador,
        p.uf,
        p.admissao AS join_date,
        v.data_venda,
        v.idVenda,
        v.idLoja,
        v.idSupervisor
    FROM jovi_lakehouse.bronze.promotores p
    LEFT JOIN jovi_lakehouse.silver.vendas v
        ON v.idPromotor = p.id
    WHERE p.status = 'AVAILABLE'
      AND p.lider <> 'TRUE'
),

aggregated AS (
    SELECT
        b.id,
        b.nome_colaborador,
        b.uf,
        b.join_date,

        COUNT(DISTINCT CASE
            WHEN b.data_venda BETWEEN p.start_last_15_days AND p.today
            THEN b.idVenda
        END) AS sales_last_15_days,

        COUNT(DISTINCT CASE
            WHEN b.data_venda BETWEEN p.start_last_30_days AND p.today
            THEN b.idVenda
        END) AS sales_last_30_days,

        COUNT(DISTINCT CASE
            WHEN b.data_venda BETWEEN p.start_last_60_days AND p.today
            THEN b.idVenda
        END) AS sales_last_60_days,

        COUNT(DISTINCT CASE
            WHEN b.data_venda BETWEEN p.start_last_90_days AND p.today
            THEN b.idVenda
        END) AS sales_last_90_days

    FROM base b
    CROSS JOIN parameters p

    GROUP BY
        b.id,
        b.nome_colaborador,
        b.uf,
        b.join_date
),

sales_last_15_by_store AS (
    SELECT
        v.idPromotor,
        v.idSupervisor,
        v.idLoja,
        l.nome_loja,
        l.rede_loja,

        COUNT(DISTINCT v.idVenda) AS sales_last_15_days_store,

        ROW_NUMBER() OVER (
            PARTITION BY v.idPromotor
            ORDER BY
                COUNT(DISTINCT v.idVenda) DESC,
                v.idLoja
        ) AS rn

    FROM jovi_lakehouse.silver.vendas v

    INNER JOIN jovi_lakehouse.bronze.lojas l
        ON l.idLoja = v.idLoja

    CROSS JOIN parameters p

    WHERE v.data_venda BETWEEN p.start_last_15_days AND p.today

    GROUP BY
        v.idPromotor,
        v.idSupervisor,
        v.idLoja,
        l.nome_loja,
        l.rede_loja
),

top_store_last_15_days AS (
    SELECT
        idPromotor,
        idSupervisor,
        idLoja,
        nome_loja,
        rede_loja
    FROM sales_last_15_by_store
    WHERE rn = 1
),

promoter_enriched AS (
    SELECT
        a.id,
        a.join_date,
        a.nome_colaborador,
        a.uf,

        t.idSupervisor,
        t.idLoja,
        t.nome_loja,
        t.rede_loja,

        a.sales_last_15_days,
        a.sales_last_30_days,
        a.sales_last_60_days,
        a.sales_last_90_days,

        ROUND(a.sales_last_15_days / 15.0, 2) AS pace_last_15_days,
        ROUND(a.sales_last_30_days / 30.0, 2) AS pace_last_30_days,
        ROUND(a.sales_last_60_days / 60.0, 2) AS pace_last_60_days,
        ROUND(a.sales_last_90_days / 90.0, 2) AS pace_last_90_days,

        ROUND(
            (
                (a.sales_last_30_days / 30.0) +
                (a.sales_last_60_days / 60.0) +
                (a.sales_last_90_days / 90.0)
            ) / 3,
            2
        ) AS avg_historical_pace

    FROM aggregated a

    LEFT JOIN top_store_last_15_days t
        ON a.id = t.idPromotor
),

supervisor_benchmark AS (
    SELECT
        idSupervisor,

        ROUND(AVG(pace_last_15_days), 2)
            AS avg_supervisor_pace_last_15_days,

        ROUND(AVG(pace_last_30_days), 2)
            AS avg_supervisor_pace_last_30_days,

        ROUND(AVG(pace_last_60_days), 2)
            AS avg_supervisor_pace_last_60_days,

        ROUND(AVG(pace_last_90_days), 2)
            AS avg_supervisor_pace_last_90_days,

        ROUND(AVG(avg_historical_pace), 2)
            AS avg_supervisor_historical_pace

    FROM promoter_enriched

    WHERE idSupervisor IS NOT NULL

    GROUP BY idSupervisor
)

SELECT
    pe.id AS idPromotor,
    pe.join_date,
    pe.nome_colaborador AS promoter_name,
    pe.uf AS state,

    pe.idSupervisor,
    s.nome_supervisor AS supervisor_name,

    pe.idLoja AS top_store_id_last_15_days,
    pe.nome_loja AS top_store_last_15_days,
    pe.rede_loja AS retailer,

    pe.sales_last_15_days,
    pe.pace_last_15_days,
    COALESCE(sb.avg_supervisor_pace_last_15_days, 0)
        AS avg_supervisor_pace_last_15_days,

    pe.sales_last_30_days,
    pe.pace_last_30_days,
    COALESCE(sb.avg_supervisor_pace_last_30_days, 0)
        AS avg_supervisor_pace_last_30_days,

    pe.sales_last_60_days,
    pe.pace_last_60_days,
    COALESCE(sb.avg_supervisor_pace_last_60_days, 0)
        AS avg_supervisor_pace_last_60_days,

    pe.sales_last_90_days,
    pe.pace_last_90_days,
    COALESCE(sb.avg_supervisor_pace_last_90_days, 0)
        AS avg_supervisor_pace_last_90_days,

    pe.avg_historical_pace,
    COALESCE(sb.avg_supervisor_historical_pace, 0)
        AS avg_supervisor_historical_pace,

    CASE
        WHEN pe.pace_last_15_days > pe.avg_historical_pace
            THEN 'GROWING'

        WHEN pe.pace_last_15_days < pe.avg_historical_pace
            THEN 'DECREASING'

        ELSE 'STABLE'
    END AS productivity_trend,

    CURRENT_TIMESTAMP() AS gold_updated_at

FROM promoter_enriched pe

LEFT JOIN jovi_lakehouse.bronze.supervisores s
    ON pe.idSupervisor = s.idSupervisor

LEFT JOIN supervisor_benchmark sb
    ON pe.idSupervisor = sb.idSupervisor;